# Notebook 15: Capstone — Full EKF-SLAM Pipeline with Interactive Tuning

This capstone notebook ties together everything from the earlier notebooks:
- simulator/world setup,
- EKF-SLAM predict/update loop,
- visualization toggles,
- tuning experiments,
- and debugging experiments that deliberately break assumptions.

> **Design goal:** safe defaults, deterministic runs, and runnable top-to-bottom with no user interaction required.


## Table of Contents

- [Learning objectives](#learning-objectives)
- [1) Load world + simulator](#1-load-world--simulator)
- [2) Create EKFSLAM with motion + measurement models](#2-create-ekfslam-with-motion--measurement-models)
- [3) Run loop step-by-step with plots](#3-run-loop-step-by-step-with-plots)
- [4) Toggle: show/hide ground truth, covariance, innovations](#4-toggle-showhide-ground-truth-covariance-innovations)
- [5) Tuning lab](#5-tuning-lab)
  - [Suggested tuning experiments](#suggested-tuning-experiments)
- [6) Debug lab](#6-debug-lab)
  - [A) Deliberately break angle wrapping and observe divergence](#a-deliberately-break-angle-wrapping-and-observe-divergence)
  - [B) Deliberately mis-set `R` and inspect NIS inconsistency](#b-deliberately-mis-set-r-and-inspect-nis-inconsistency)
- [Exercises (with suggested experiments)](#exercises-with-suggested-experiments)
- [Recap](#recap)

---

## Navigation

- Previous: [ntbk-14-ekf-slam-state-augmentation.ipynb](./ntbk-14-ekf-slam-state-augmentation.ipynb)
- Next: [ntbk-zero-to-hero-kiss-slam.ipynb](./ntbk-zero-to-hero-kiss-slam.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/ekf_slam.py`
- `src/kiss_slam/sim/simulator.py`
- `src/kiss_slam/viz/live_viewer.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 114
set_seed(SEED)
plt.rcdefaults()


## Learning objectives

By the end of this notebook, you should be able to:
1. Build a complete EKF-SLAM pipeline from world generation to evaluation.
2. Run and visualize the step-by-step filtering loop.
3. Toggle diagnostics (ground truth, covariance ellipses, innovation vectors).
4. Tune process/measurement noise and simulator realism settings.
5. Diagnose divergence caused by angle-wrapping bugs and inconsistent noise assumptions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from kiss_slam.ekf_slam import EKFSLAM
from kiss_slam.math_utils import mahalanobis_distance, wrap_angle
from kiss_slam.models.motion import DifferentialDriveMotionModel
from kiss_slam.models.measurement import RangeBearingMeasurementModel
from kiss_slam.sim.world import World2D
from kiss_slam.sim.simulator import SimConfig, Simulator
from kiss_slam.types import EKFSLAMConfig

np.random.seed(15)


## 1) Load world + simulator

In [ ]:
SAFE_DEFAULTS = {
    "seed": 15,
    "n_landmarks": 24,
    "xlim": (-20.0, 20.0),
    "ylim": (-20.0, 20.0),
    "steps": 180,
    "trajectory_mode": "figure_eight",
    "process_noise_std": (0.05, 0.03),
    "measurement_noise_std": (0.20, np.deg2rad(3.0)),
    "dropout": 0.05,
    "sensor_range": 13.0,
    "fov_rad": np.pi,
}

def make_world_and_sim(*, seed: int, n_landmarks: int, steps: int, process_noise_std: tuple[float, float], measurement_noise_std: tuple[float, float], dropout: float, trajectory_mode: str = "figure_eight"):
    world = World2D.pattern(
        pattern="random",
        n_landmarks=n_landmarks,
        xlim=SAFE_DEFAULTS["xlim"],
        ylim=SAFE_DEFAULTS["ylim"],
        seed=seed,
    )

    sim_cfg = SimConfig(
        dt=0.1,
        sensor_range=SAFE_DEFAULTS["sensor_range"],
        fov_rad=SAFE_DEFAULTS["fov_rad"],
        trajectory_mode=trajectory_mode,
        control_noise_std=process_noise_std,
        measurement_noise_std=measurement_noise_std,
        measurement_dropout_prob=dropout,
        steps=steps,
    )
    sim = Simulator(world=world, config=sim_cfg, seed=seed)
    return world, sim

world, sim = make_world_and_sim(
    seed=SAFE_DEFAULTS["seed"],
    n_landmarks=SAFE_DEFAULTS["n_landmarks"],
    steps=SAFE_DEFAULTS["steps"],
    process_noise_std=SAFE_DEFAULTS["process_noise_std"],
    measurement_noise_std=SAFE_DEFAULTS["measurement_noise_std"],
    dropout=SAFE_DEFAULTS["dropout"],
)

print(f"Loaded world with {len(world.landmarks)} landmarks.")
print(f"Simulator steps: {sim.config.steps}, dt: {sim.config.dt:.2f}s")


## 2) Create EKFSLAM with motion + measurement models

In [ ]:
def make_ekf(process_noise_std, measurement_noise_std):
    q = np.diag(np.square(process_noise_std))
    r = np.diag(np.square(measurement_noise_std))
    cfg = EKFSLAMConfig(process_noise=q, measurement_noise=r)
    return EKFSLAM(
        motion_model=DifferentialDriveMotionModel(),
        measurement_model=RangeBearingMeasurementModel(),
        config=cfg,
    )

ekf = make_ekf(
    process_noise_std=SAFE_DEFAULTS["process_noise_std"],
    measurement_noise_std=SAFE_DEFAULTS["measurement_noise_std"],
)
print("EKF-SLAM initialized.")


## 3) Run loop step-by-step with plots

In [ ]:
def covariance_ellipse_points(cov2: np.ndarray, center: np.ndarray, n_sigma: float = 2.0, n_pts: int = 100):
    eigvals, eigvecs = np.linalg.eigh(cov2)
    eigvals = np.clip(eigvals, 1e-12, None)
    t = np.linspace(0.0, 2.0 * np.pi, n_pts)
    unit = np.vstack((np.cos(t), np.sin(t)))
    scale = np.diag(n_sigma * np.sqrt(eigvals))
    pts = eigvecs @ scale @ unit
    return pts.T + center[None, :]

def run_pipeline(world, sim, ekf, steps: int):
    true_traj = []
    est_traj = []
    innovation_segments = []

    for _ in range(steps):
        u, dt, measurements, gt_pose = sim.step()
        ekf.step(u=u, dt=dt, measurements=measurements)

        true_traj.append(gt_pose.as_array())
        est_traj.append(ekf.robot_pose())
        innovation_segments.extend(ekf.innovation_vectors())

    return {
        "true_traj": np.array(true_traj),
        "est_traj": np.array(est_traj),
        "landmarks_est": ekf.landmark_states(),
        "state_cov": ekf.get_covariance(),
        "innovation_segments": innovation_segments,
        "nis_values": np.array(ekf.nis_values, dtype=float),
    }

def plot_run(result, world, *, show_ground_truth=True, show_covariance=True, show_innovations=False, title="EKF-SLAM run"):
    fig, ax = plt.subplots()

    true_xy = result["true_traj"][:, :2]
    est_xy = result["est_traj"][:, :2]

    if show_ground_truth:
        ax.plot(true_xy[:, 0], true_xy[:, 1], "-", color="tab:green", lw=1.5, label="true trajectory")
        gt_lm = np.array([[lm.x, lm.y] for lm in world.landmarks], dtype=float)
        ax.scatter(gt_lm[:, 0], gt_lm[:, 1], marker="x", color="tab:gray", alpha=0.7, label="true landmarks")

    ax.plot(est_xy[:, 0], est_xy[:, 1], "-", color="tab:blue", lw=1.8, label="estimated trajectory")

    est_landmarks = result["landmarks_est"]
    if est_landmarks:
        est_arr = np.array(list(est_landmarks.values()), dtype=float)
        ax.scatter(est_arr[:, 0], est_arr[:, 1], marker="o", facecolors="none", edgecolors="tab:red", label="estimated landmarks")

    if show_covariance:
        cov = result["state_cov"]
        robot_xy = result["est_traj"][-1, :2]
        if cov.shape[0] >= 2:
            robot_ell = covariance_ellipse_points(cov[:2, :2], robot_xy, n_sigma=2.0)
            ax.plot(robot_ell[:, 0], robot_ell[:, 1], "--", color="tab:orange", lw=1.5, label="robot 2σ ellipse")

    if show_innovations and result["innovation_segments"]:
        seg = result["innovation_segments"]
        for start, end in seg[-120:]:
            ax.plot([start[0], end[0]], [start[1], end[1]], color="tab:purple", alpha=0.25, lw=0.8)

    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(title)
    ax.legend(loc="best")
    plt.show()


In [ ]:
sim.reset()
ekf = make_ekf(SAFE_DEFAULTS["process_noise_std"], SAFE_DEFAULTS["measurement_noise_std"])

result_default = run_pipeline(world, sim, ekf, steps=SAFE_DEFAULTS["steps"])
plot_run(result_default, world, title="Capstone default run")

print(f"Final pose error norm: {np.linalg.norm(result_default['true_traj'][-1,:2] - result_default['est_traj'][-1,:2]):.3f} m")
print(f"NIS samples collected: {len(result_default['nis_values'])}")


## 4) Toggle: show/hide ground truth, covariance, innovations

In [ ]:
# Toggle these booleans and re-run this cell if you want different overlays.
SHOW_GROUND_TRUTH = True
SHOW_COVARIANCE = True
SHOW_INNOVATIONS = True

plot_run(
    result_default,
    world,
    show_ground_truth=SHOW_GROUND_TRUTH,
    show_covariance=SHOW_COVARIANCE,
    show_innovations=SHOW_INNOVATIONS,
    title="Diagnostics toggle view",
)


## 5) Tuning lab

We will compare multiple settings:
- process noise and measurement noise,
- number of landmarks,
- measurement dropout / noise severity.

We report quick scalar metrics:
- final position error,
- trajectory RMSE,
- NIS inlier ratio using rough 95% bounds for 2 DoF (`0.103` to `5.991`).


In [ ]:
def trajectory_rmse(true_traj: np.ndarray, est_traj: np.ndarray) -> float:
    d = true_traj[:, :2] - est_traj[:, :2]
    return float(np.sqrt(np.mean(np.sum(d * d, axis=1))))

def nis_inlier_ratio(nis_values: np.ndarray, low: float = 0.103, high: float = 5.991) -> float:
    if nis_values.size == 0:
        return np.nan
    inside = (nis_values >= low) & (nis_values <= high)
    return float(np.mean(inside))

def run_scenario(name, *, n_landmarks, process_noise_std, measurement_noise_std, dropout, steps=160, seed=21):
    world_i, sim_i = make_world_and_sim(
        seed=seed,
        n_landmarks=n_landmarks,
        steps=steps,
        process_noise_std=process_noise_std,
        measurement_noise_std=measurement_noise_std,
        dropout=dropout,
    )
    ekf_i = make_ekf(process_noise_std, measurement_noise_std)
    sim_i.reset()
    out = run_pipeline(world_i, sim_i, ekf_i, steps=steps)

    final_err = float(np.linalg.norm(out["true_traj"][-1, :2] - out["est_traj"][-1, :2]))
    rmse = trajectory_rmse(out["true_traj"], out["est_traj"])
    inlier = nis_inlier_ratio(out["nis_values"])

    return {
        "name": name,
        "result": out,
        "world": world_i,
        "final_pos_error": final_err,
        "traj_rmse": rmse,
        "nis_inlier_ratio": inlier,
    }

scenarios = [
    dict(name="safe-default", n_landmarks=24, process_noise_std=(0.05, 0.03), measurement_noise_std=(0.20, np.deg2rad(3.0)), dropout=0.05),
    dict(name="optimistic-noise", n_landmarks=24, process_noise_std=(0.02, 0.01), measurement_noise_std=(0.10, np.deg2rad(1.5)), dropout=0.02),
    dict(name="harsh-noise+dropout", n_landmarks=24, process_noise_std=(0.10, 0.07), measurement_noise_std=(0.40, np.deg2rad(6.0)), dropout=0.25),
    dict(name="few-landmarks", n_landmarks=10, process_noise_std=(0.05, 0.03), measurement_noise_std=(0.20, np.deg2rad(3.0)), dropout=0.05),
    dict(name="many-landmarks", n_landmarks=40, process_noise_std=(0.05, 0.03), measurement_noise_std=(0.20, np.deg2rad(3.0)), dropout=0.05),
]

reports = [run_scenario(**cfg) for cfg in scenarios]
for rep in reports:
    print(
        f"{rep['name']:<18} | final_err={rep['final_pos_error']:.3f} m"
        f" | rmse={rep['traj_rmse']:.3f} m"
        f" | NIS_inlier_ratio={rep['nis_inlier_ratio']:.2f}"
    )


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = [r["name"] for r in reports]
rmse_vals = [r["traj_rmse"] for r in reports]
nis_vals = [r["nis_inlier_ratio"] for r in reports]

axes[0].bar(labels, rmse_vals, color="tab:blue", alpha=0.8)
axes[0].set_title("Trajectory RMSE")
axes[0].tick_params(axis="x", rotation=35)
axes[0].set_ylabel("meters")

axes[1].bar(labels, nis_vals, color="tab:orange", alpha=0.8)
axes[1].set_title("NIS Inlier Ratio (95% window)")
axes[1].tick_params(axis="x", rotation=35)
axes[1].set_ylim(0.0, 1.0)

plt.tight_layout()
plt.show()


### Suggested tuning experiments

1. Increase only **bearing noise** while keeping range noise fixed. What degrades first: heading or map geometry?
2. Keep noise fixed and sweep **dropout** from `0.0` to `0.5`. Watch when the filter loses map lock.
3. Keep simulator fixed and intentionally set EKF `Q` or `R` too small (overconfident) vs too large (underconfident).
4. Compare `n_landmarks=8` vs `n_landmarks=50` for the same trajectory.


## 6) Debug lab

### A) Deliberately break angle wrapping and observe divergence

Below we subclass EKFSLAM and apply an intentionally wrong angle-wrapping rule for the bearing innovation.
This creates discontinuities near ±π and often causes unstable updates.


In [ ]:
class BrokenAngleEKF(EKFSLAM):
    """Intentionally broken EKF update: bearing innovation is NOT wrapped."""

    def _update_landmark(self, landmark_id, measurement, measurement_cov):
        start = self._landmark_index[landmark_id]
        robot_state = self._mu[:3]
        landmark_state = self._mu[start : start + 2]

        z = measurement.as_array()
        z_hat = self.measurement_model.predict(robot_state, landmark_state)
        innovation = z - z_hat
        # BUG: deliberately wrong "wrapping" into [0, 2π), which creates
        # large discontinuities near the branch cut.
        innovation[1] = np.mod(innovation[1], 2.0 * np.pi)

        hr, hl = self.measurement_model.jacobians(robot_state, landmark_state)
        n = self._mu.size
        h = np.zeros((2, n), dtype=float)
        h[:, :3] = hr
        h[:, start : start + 2] = hl

        s = h @ self._sigma @ h.T + measurement_cov
        k = self._sigma @ h.T @ np.linalg.inv(s)

        self._mu = self._mu + k @ innovation
        self._mu[2] = wrap_angle(self._mu[2])

        i_n = np.eye(n, dtype=float)
        residual = i_n - k @ h
        self._sigma = residual @ self._sigma @ residual.T + k @ measurement_cov @ k.T

        self._nis_values.append(mahalanobis_distance(innovation, s))

def run_debug_angle_wrap(seed=33):
    world_d, sim_d = make_world_and_sim(
        seed=seed,
        n_landmarks=24,
        steps=180,
        process_noise_std=(0.05, 0.03),
        measurement_noise_std=(0.20, np.deg2rad(3.0)),
        dropout=0.05,
    )

    sim_d.reset()
    ekf_ok = make_ekf((0.05, 0.03), (0.20, np.deg2rad(3.0)))
    out_ok = run_pipeline(world_d, sim_d, ekf_ok, 180)

    sim_d.reset()
    ekf_bad = BrokenAngleEKF(
        motion_model=DifferentialDriveMotionModel(),
        measurement_model=RangeBearingMeasurementModel(),
        config=EKFSLAMConfig(
            process_noise=np.diag(np.square((0.05, 0.03))),
            measurement_noise=np.diag(np.square((0.20, np.deg2rad(3.0)))),
        ),
    )
    out_bad = run_pipeline(world_d, sim_d, ekf_bad, 180)

    return world_d, out_ok, out_bad

world_dbg, out_ok, out_bad = run_debug_angle_wrap()

err_ok = trajectory_rmse(out_ok["true_traj"], out_ok["est_traj"])
err_bad = trajectory_rmse(out_bad["true_traj"], out_bad["est_traj"])
print(f"RMSE (correct wrapping): {err_ok:.3f} m")
print(f"RMSE (broken wrapping):  {err_bad:.3f} m")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(out_ok["true_traj"][:, 0], out_ok["true_traj"][:, 1], color="tab:green", lw=2.0, label="ground truth")
ax.plot(out_ok["est_traj"][:, 0], out_ok["est_traj"][:, 1], "--", color="tab:blue", label="EKF (correct)")
ax.plot(out_bad["est_traj"][:, 0], out_bad["est_traj"][:, 1], "-.", color="tab:red", label="EKF (broken angle wrap)")
ax.set_aspect("equal")
ax.set_title("Debug lab: angle wrapping bug causes divergence")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.legend(loc="best")
plt.show()


### B) Deliberately mis-set `R` and inspect NIS inconsistency

We keep simulator noise fixed, but run EKF with:
- a **consistent** `R` (matches simulator),
- an **underestimated** `R` (too small, overconfident).

When `R` is underestimated, NIS tends to be too high and inlier ratio drops.


In [ ]:
def run_debug_R(seed=44):
    process_std = (0.05, 0.03)
    true_meas_std = (0.20, np.deg2rad(3.0))

    world_r, sim_r = make_world_and_sim(
        seed=seed,
        n_landmarks=24,
        steps=180,
        process_noise_std=process_std,
        measurement_noise_std=true_meas_std,
        dropout=0.05,
    )

    sim_r.reset()
    ekf_consistent = make_ekf(process_std, true_meas_std)
    out_consistent = run_pipeline(world_r, sim_r, ekf_consistent, 180)

    sim_r.reset()
    underestimated_R = (0.08, np.deg2rad(1.0))
    ekf_overconfident = make_ekf(process_std, underestimated_R)
    out_overconfident = run_pipeline(world_r, sim_r, ekf_overconfident, 180)

    return out_consistent, out_overconfident

out_consistent, out_overconfident = run_debug_R()

ratio_consistent = nis_inlier_ratio(out_consistent["nis_values"])
ratio_overconfident = nis_inlier_ratio(out_overconfident["nis_values"])

print(f"NIS inlier ratio (consistent R):    {ratio_consistent:.2f}")
print(f"NIS inlier ratio (underestimated R): {ratio_overconfident:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(out_consistent["nis_values"], bins=35, alpha=0.65, label="consistent R", color="tab:blue", density=True)
ax.hist(out_overconfident["nis_values"], bins=35, alpha=0.55, label="underestimated R", color="tab:red", density=True)
ax.axvline(5.991, color="k", ls="--", lw=1.2, label="95% upper bound (2 DoF)")
ax.set_xlim(0, max(10, np.percentile(out_overconfident["nis_values"], 95)))
ax.set_xlabel("NIS")
ax.set_ylabel("density")
ax.set_title("Debug lab: NIS inconsistency with mis-set R")
ax.legend(loc="best")
plt.show()


## Exercises (with suggested experiments)

1. **Tuning challenge:** Find a parameter set with trajectory RMSE < 0.5 m and NIS inlier ratio between 0.6 and 0.9.
2. **Stress test:** Increase dropout to 0.4 and reduce landmarks to 8. What fails first?
3. **Model mismatch:** Keep simulator as-is but set EKF process noise 5× too small. Compare trajectories and NIS.
4. **Bug hunt:** In `BrokenAngleEKF`, re-introduce angle wrapping and confirm performance recovers.

<details>
<summary>Optional solution hints</summary>

- Start from safe defaults; change one variable at a time.
- If NIS is too high, increase EKF `R` and/or `Q`.
- If NIS is too low, your filter is probably too conservative.
- Angle wrapping should be applied to bearing innovation before the Kalman update.

</details>


## Recap

You now have a full, reproducible EKF-SLAM capstone pipeline that includes:
- deterministic world + simulator generation,
- end-to-end predict/update execution,
- diagnostics visualization toggles,
- practical tuning workflows,
- and debugging patterns for common EKF pitfalls.


## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
